In [1]:
# 🔹 CELDA 1: Introducción a Attention

import torch
import torch.nn as nn
import math
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification

print("=" * 60)
print("SEMANA 11: TRANSFORMERS & ATTENTION MECHANISM")
print("=" * 60)

print("""
¿QUÉ HACE ESTA CELDA?
Explica el corazón de Transformers: ATTENTION.

ANTES (RNN/LSTM):
- Procesa palabra por palabra
- Memoria limitada (vanishing gradient)
- Lento (secuencial)

AHORA (Transformers):
- Ve TODO el texto simultáneamente
- Atención ponderada entre palabras
- Rápido (paralelo)

SELF-ATTENTION:
"El gato subió al árbol porque tenía hambre"
          ↑                                 ↑
      ATENCIÓN: "gato" se conecta con "tenía hambre"

FÓRMULA:
Attention(Q, K, V) = softmax(Q @ K^T / √d) @ V

Q = Query (¿qué busco?)
K = Key (¿qué información tienes?)
V = Value (dame la información)
""")

print("""
VENTAJAS:
✓ Parallelizable (procesa todo a la vez)
✓ Long-range dependencies (conecta palabras lejanas)
✓ Interpretable (ves qué atiende a qué)
✓ State-of-the-art (BERT, GPT, etc)

TRANSFORMERS USAN:
1. Self-Attention (intra-token)
2. Multi-Head Attention (múltiples perspectivas)
3. Feed-Forward Networks (procesamiento)
4. LayerNorm y Residual connections (estabilidad)
""")


SEMANA 11: TRANSFORMERS & ATTENTION MECHANISM

¿QUÉ HACE ESTA CELDA?
Explica el corazón de Transformers: ATTENTION.

ANTES (RNN/LSTM):
- Procesa palabra por palabra
- Memoria limitada (vanishing gradient)
- Lento (secuencial)

AHORA (Transformers):
- Ve TODO el texto simultáneamente
- Atención ponderada entre palabras
- Rápido (paralelo)

SELF-ATTENTION:
"El gato subió al árbol porque tenía hambre"
          ↑                                 ↑
      ATENCIÓN: "gato" se conecta con "tenía hambre"

FÓRMULA:
Attention(Q, K, V) = softmax(Q @ K^T / √d) @ V

Q = Query (¿qué busco?)
K = Key (¿qué información tienes?)
V = Value (dame la información)


VENTAJAS:
✓ Parallelizable (procesa todo a la vez)
✓ Long-range dependencies (conecta palabras lejanas)
✓ Interpretable (ves qué atiende a qué)
✓ State-of-the-art (BERT, GPT, etc)

TRANSFORMERS USAN:
1. Self-Attention (intra-token)
2. Multi-Head Attention (múltiples perspectivas)
3. Feed-Forward Networks (procesamiento)
4. LayerNorm y Residual co

In [2]:
# 🔹 CELDA 2: Implementar Self-Attention

print("\n" + "=" * 60)
print("SELF-ATTENTION DESDE CERO")
print("=" * 60)

print("""
¿QUÉ HACE ESTA CELDA?
Implementa Self-Attention manualmente.
(PyTorch lo optimiza, pero esto es para ENTENDER)
""")

class SelfAttention(nn.Module):
    def __init__(self, d_model):
        """
        d_model: dimensión de embeddings
        """
        super(SelfAttention, self).__init__()
        self.d_model = d_model
        
        # Proyecciones lineales para Q, K, V
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        
        self.softmax = nn.Softmax(dim=-1)
    
    def forward(self, x):
        """
        ¿QUÉ HACE?
        x: (batch, seq_len, d_model)
        
        1. Proyecta a Q, K, V
        2. Calcula scores: Q @ K^T / √d
        3. Softmax para pesos
        4. Multiplica por V
        
        Resultado: Cada token ve a todos los otros (con pesos)
        """
        # Proyectar
        Q = self.W_q(x)  # (batch, seq_len, d_model)
        K = self.W_k(x)
        V = self.W_v(x)
        
        # Scores: Q @ K^T / √d
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_model)
        # (batch, seq_len, seq_len)
        
        # Weights: softmax
        weights = self.softmax(scores)  # Suma a 1 en última dim
        # (batch, seq_len, seq_len)
        
        # Apply attention: weights @ V
        output = torch.matmul(weights, V)
        # (batch, seq_len, d_model)
        
        return output, weights

# Test
d_model = 8
seq_len = 4
batch_size = 2

# Input: 2 muestras, 4 tokens, 8 dimensiones
x = torch.randn(batch_size, seq_len, d_model)

attention = SelfAttention(d_model)
output, weights = attention(x)

print(f"\n📊 Self-Attention:")
print(f"  Input: {x.shape}")
print(f"  Output: {output.shape}")
print(f"  Attention weights: {weights.shape}")

print(f"\n🔍 Ejemplo: Matriz de atención (primera muestra):")
print(f"  (qué token atiende a cuál)")
print(weights[0].detach().numpy().round(3))

print(f"\n💡 Interpretación:")
print(f"  Row 0 (token 0) atiende a: {weights[0, 0].detach().numpy().round(3)}")
print(f"  → suma 1.0 (distribución de probabilidad)")



SELF-ATTENTION DESDE CERO

¿QUÉ HACE ESTA CELDA?
Implementa Self-Attention manualmente.
(PyTorch lo optimiza, pero esto es para ENTENDER)


📊 Self-Attention:
  Input: torch.Size([2, 4, 8])
  Output: torch.Size([2, 4, 8])
  Attention weights: torch.Size([2, 4, 4])

🔍 Ejemplo: Matriz de atención (primera muestra):
  (qué token atiende a cuál)
[[0.258 0.26  0.331 0.15 ]
 [0.249 0.224 0.448 0.08 ]
 [0.225 0.241 0.301 0.233]
 [0.195 0.141 0.462 0.202]]

💡 Interpretación:
  Row 0 (token 0) atiende a: [0.258 0.26  0.331 0.15 ]
  → suma 1.0 (distribución de probabilidad)


In [3]:
# 🔹 CELDA 3: Multi-Head Attention

print("\n" + "=" * 60)
print("MULTI-HEAD ATTENTION")
print("=" * 60)

print("""
¿QUÉ HACE ESTA CELDA?
Self-Attention con múltiples "cabezas" (perspectivas).

¿POR QUÉ?
- 1 attention: ve solo 1 patrón
- 8 attentions: ve 8 patrones simultáneamente

EJEMPLO:
Head 1: Atiende a "sustantivos"
Head 2: Atiende a "verbos"
Head 3: Atiende a "adjetivos"
...
Head 8: Atiende a "dependencias distantes"

Luego combina todo.
""")

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        """
        d_model: dimensión total (ej: 512)
        num_heads: número de cabezas (ej: 8)
        """
        super(MultiHeadAttention, self).__init__()
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # Dimensión por cabeza
        
        # Proyecciones para Q, K, V
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        
        # Output
        self.W_o = nn.Linear(d_model, d_model)
        
        self.softmax = nn.Softmax(dim=-1)
    
    def forward(self, x):
        """
        ¿QUÉ HACE?
        1. Divide Q, K, V en num_heads partes
        2. Aplica attention en cada cabeza
        3. Concatena resultados
        4. Proyecta output
        """
        batch_size, seq_len, d_model = x.shape
        
        # Proyectar
        Q = self.W_q(x)  # (batch, seq_len, d_model)
        K = self.W_k(x)
        V = self.W_v(x)
        
        # Split en heads
        # (batch, seq_len, d_model) → (batch, seq_len, num_heads, d_k)
        Q = Q.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        # (batch, num_heads, seq_len, d_k)
        K = K.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        
        # Attention por head
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        weights = self.softmax(scores)
        head_outputs = torch.matmul(weights, V)
        # (batch, num_heads, seq_len, d_k)
        
        # Concatenar heads
        output = head_outputs.transpose(1, 2).contiguous()
        # (batch, seq_len, num_heads, d_k)
        output = output.view(batch_size, seq_len, d_model)
        # (batch, seq_len, d_model)
        
        # Proyectar output
        output = self.W_o(output)
        
        return output

# Test
d_model = 16
num_heads = 4
seq_len = 5
batch_size = 2

x = torch.randn(batch_size, seq_len, d_model)

mha = MultiHeadAttention(d_model, num_heads)
output = mha(x)

print(f"\n📊 Multi-Head Attention:")
print(f"  Input: {x.shape}")
print(f"  Output: {output.shape}")
print(f"  Heads: {num_heads}")
print(f"  Dim por cabeza: {d_model // num_heads}")

print(f"\n✅ Cada cabeza ve patrones diferentes")
print(f"  Luego se combinan para resultado final")



MULTI-HEAD ATTENTION

¿QUÉ HACE ESTA CELDA?
Self-Attention con múltiples "cabezas" (perspectivas).

¿POR QUÉ?
- 1 attention: ve solo 1 patrón
- 8 attentions: ve 8 patrones simultáneamente

EJEMPLO:
Head 1: Atiende a "sustantivos"
Head 2: Atiende a "verbos"
Head 3: Atiende a "adjetivos"
...
Head 8: Atiende a "dependencias distantes"

Luego combina todo.


📊 Multi-Head Attention:
  Input: torch.Size([2, 5, 16])
  Output: torch.Size([2, 5, 16])
  Heads: 4
  Dim por cabeza: 4

✅ Cada cabeza ve patrones diferentes
  Luego se combinan para resultado final


In [4]:
# 🔹 CELDA 4: Transformer Block Completo

print("\n" + "=" * 60)
print("TRANSFORMER BLOCK (Componente real)")
print("=" * 60)

print("""
¿QUÉ HACE ESTA CELDA?
Implementa un bloque Transformer completo.
(Esto es lo que usa BERT, GPT, etc)

ESTRUCTURA:
1. Multi-Head Attention
2. Feed-Forward Network
3. Layer Normalization + Residual connections
4. Dropout (regularización)
""")

class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1):
        super(TransformerBlock, self).__init__()
        
        # Multi-Head Attention
        self.attention = MultiHeadAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model)
        
        # Feed-Forward Network
        self.ff = nn.Sequential(
            nn.Linear(d_model, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, d_model)
        )
        self.norm2 = nn.LayerNorm(d_model)
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        """
        ¿QUÉ HACE?
        1. Attention + Residual + Norm
        2. Feed-Forward + Residual + Norm
        3. Dropout para regularización
        
        Residual: x_out = x_in + transform(x_in)
        (Permite que información pase sin cambios)
        """
        # Self-Attention
        attn_output = self.attention(x)
        attn_output = self.dropout(attn_output)
        x = self.norm1(x + attn_output)  # Residual + Norm
        
        # Feed-Forward
        ff_output = self.ff(x)
        ff_output = self.dropout(ff_output)
        x = self.norm2(x + ff_output)  # Residual + Norm
        
        return x

# Test
d_model = 16
num_heads = 4
ff_dim = 64
seq_len = 5
batch_size = 2

x = torch.randn(batch_size, seq_len, d_model)

block = TransformerBlock(d_model, num_heads, ff_dim)
output = block(x)

print(f"\n📊 Transformer Block:")
print(f"  Input: {x.shape}")
print(f"  Output: {output.shape}")

total_params = sum(p.numel() for p in block.parameters())
print(f"  Parámetros: {total_params:,}")

print(f"\n✅ Este bloque apila N veces → Transformer")
print(f"  BERT: 12 bloques")
print(f"  GPT-2: 12 bloques")
print(f"  GPT-3: 96 bloques")



TRANSFORMER BLOCK (Componente real)

¿QUÉ HACE ESTA CELDA?
Implementa un bloque Transformer completo.
(Esto es lo que usa BERT, GPT, etc)

ESTRUCTURA:
1. Multi-Head Attention
2. Feed-Forward Network
3. Layer Normalization + Residual connections
4. Dropout (regularización)


📊 Transformer Block:
  Input: torch.Size([2, 5, 16])
  Output: torch.Size([2, 5, 16])
  Parámetros: 3,280

✅ Este bloque apila N veces → Transformer
  BERT: 12 bloques
  GPT-2: 12 bloques
  GPT-3: 96 bloques


In [5]:
# 🔹 CELDA 5: Fine-tune BERT Preentrenado

print("\n" + "=" * 60)
print("BERT PREENTRENADO: Fine-tuning para clasificación")
print("=" * 60)

print("""
¿QUÉ HACE ESTA CELDA?
Carga BERT (preentrenado en millones de textos).
Lo ajustas (fine-tune) para clasificación de sentimientos.

BERT:
- Entrenado en 3B palabras (Wikipedia + BookCorpus)
- Aprende representaciones profundas de lenguaje
- 340M parámetros
- State-of-the-art en 2018

¿POR QUÉ fine-tune?
- No reentrenar desde 0 (toma meses)
- Ajustar los últimos 10% de parámetros (toma horas)
- Transfer learning: usa conocimiento previo
""")

# Simular BERT (en prod usarías transformers library)
print("\n🏗️ Simulación de BERT fine-tuning:")

class BERTClassifier(nn.Module):
    def __init__(self, d_model=768, num_classes=2):
        super(BERTClassifier, self).__init__()
        
        # Simular BERT encoder (en prod: from transformers import BertModel)
        # Aquí simplificado: transformer blocks
        self.encoder = nn.Sequential(
            TransformerBlock(d_model, num_heads=12, ff_dim=3072),
            TransformerBlock(d_model, num_heads=12, ff_dim=3072),
        )
        
        # Cabeza de clasificación (lo que fine-tuneamos)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 256),
            nn.Dropout(0.1),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, embeddings):
        """
        embeddings: (batch, seq_len, d_model)
        output: (batch, num_classes)
        """
        # Encoder: Transformer blocks (frozen en fine-tuning real)
        encoded = self.encoder(embeddings)
        
        # Tomar [CLS] token (primer token, representa todo el texto)
        cls_output = encoded[:, 0, :]  # (batch, d_model)
        
        # Clasificación
        logits = self.classifier(cls_output)  # (batch, num_classes)
        
        return logits

# Test
d_model = 768
batch_size = 4
seq_len = 32
num_classes = 2

# Simular embeddings BERT
embeddings = torch.randn(batch_size, seq_len, d_model)

bert_classifier = BERTClassifier(d_model, num_classes)
logits = bert_classifier(embeddings)

print(f"\n📊 BERT Classifier:")
print(f"  Input embeddings: {embeddings.shape}")
print(f"  Output logits: {logits.shape}")

total_params = sum(p.numel() for p in bert_classifier.parameters())
print(f"  Parámetros: {total_params:,}")

# Predicciones
probs = torch.softmax(logits, dim=1)
predictions = torch.argmax(probs, dim=1)

print(f"\n✅ Predicciones:")
for i in range(batch_size):
    sentiment = "😊 Positivo" if predictions[i] == 1 else "😞 Negativo"
    confidence = probs[i, predictions[i]].item()
    print(f"  Muestra {i+1}: {sentiment} ({confidence:.1%} confianza)")

print(f"\n💡 En producción:")
print(f"  Usarías: from transformers import BertForSequenceClassification")
print(f"  Luego: fine-tune en 1-2 horas con GPU")



BERT PREENTRENADO: Fine-tuning para clasificación

¿QUÉ HACE ESTA CELDA?
Carga BERT (preentrenado en millones de textos).
Lo ajustas (fine-tune) para clasificación de sentimientos.

BERT:
- Entrenado en 3B palabras (Wikipedia + BookCorpus)
- Aprende representaciones profundas de lenguaje
- 340M parámetros
- State-of-the-art en 2018

¿POR QUÉ fine-tune?
- No reentrenar desde 0 (toma meses)
- Ajustar los últimos 10% de parámetros (toma horas)
- Transfer learning: usa conocimiento previo


🏗️ Simulación de BERT fine-tuning:

📊 BERT Classifier:
  Input embeddings: torch.Size([4, 32, 768])
  Output logits: torch.Size([4, 2])
  Parámetros: 14,373,122

✅ Predicciones:
  Muestra 1: 😞 Negativo (50.2% confianza)
  Muestra 2: 😞 Negativo (50.1% confianza)
  Muestra 3: 😊 Positivo (52.6% confianza)
  Muestra 4: 😞 Negativo (53.3% confianza)

💡 En producción:
  Usarías: from transformers import BertForSequenceClassification
  Luego: fine-tune en 1-2 horas con GPU


In [6]:
# 🔹 CELDA 6: Resumen Semana 11

print("""
✅ SEMANA 11: TRANSFORMERS & ATTENTION

CONCEPTOS CLAVE APRENDIDOS:
✓ Self-Attention: Conecta todos los tokens
✓ Multi-Head Attention: Múltiples perspectivas
✓ Transformer Block: Attention + Feed-Forward + Residual
✓ Transfer Learning: Pre-train + Fine-tune
✓ BERT/GPT: Arquitectura state-of-the-art

TRANSFORMERS vs RNN:
├─ Parallelizable (RNN: secuencial)
├─ Long-range dependencies (RNN: problemas)
├─ Interpretable (ves attention weights)
└─ Escalable (RNN: escala mal)

ARQUITECTURA TRANSFORMER:
Input → Embedding + Positional Encoding → 
N × [Multi-Head Attention + Feed-Forward] →
Output (clasificación/traducción/QA)

APLICACIONES REALES:
✓ BERT: 12 capas, 340M params (Google 2018)
✓ GPT-2: 12 capas, 1.5B params (OpenAI 2019)
✓ GPT-3: 96 capas, 175B params (OpenAI 2020)
✓ ChatGPT: Fine-tune de GPT-3.5
✓ Traducción, Summarización, QA

MES 1 + SEMANA 11 COMPLETADA

Dominas: Fundamentos + ML + DL + CNN + RNN + NLP + TRANSFORMERS

PRÓXIMA SEMANA:
- Semana 12: LLMs + RAG CAPSTONE
  └─ Crear chatbot completo (como ChatGPT mini)
  └─ RAG: Retrieval-Augmented Generation
  └─ Proyecto final: Case study profesional

FELICIDADES: Vas a nivel MIT/Stanford 🎓
""")



✅ SEMANA 11: TRANSFORMERS & ATTENTION

CONCEPTOS CLAVE APRENDIDOS:
✓ Self-Attention: Conecta todos los tokens
✓ Multi-Head Attention: Múltiples perspectivas
✓ Transformer Block: Attention + Feed-Forward + Residual
✓ Transfer Learning: Pre-train + Fine-tune
✓ BERT/GPT: Arquitectura state-of-the-art

TRANSFORMERS vs RNN:
├─ Parallelizable (RNN: secuencial)
├─ Long-range dependencies (RNN: problemas)
├─ Interpretable (ves attention weights)
└─ Escalable (RNN: escala mal)

ARQUITECTURA TRANSFORMER:
Input → Embedding + Positional Encoding → 
N × [Multi-Head Attention + Feed-Forward] →
Output (clasificación/traducción/QA)

APLICACIONES REALES:
✓ BERT: 12 capas, 340M params (Google 2018)
✓ GPT-2: 12 capas, 1.5B params (OpenAI 2019)
✓ GPT-3: 96 capas, 175B params (OpenAI 2020)
✓ ChatGPT: Fine-tune de GPT-3.5
✓ Traducción, Summarización, QA

MES 1 + SEMANA 11 COMPLETADA

Dominas: Fundamentos + ML + DL + CNN + RNN + NLP + TRANSFORMERS

PRÓXIMA SEMANA:
- Semana 12: LLMs + RAG CAPSTONE
  └─ Crear